In [1]:
# =============================================================================
# CELL 1 — Install Dependencies
# =============================================================================

#%pip install entsoe-py requests --quiet

#print("✅ Dependencies installed")

StatementMeta(, c2d68a54-5456-404a-8387-b3e8c4e398ac, 8, Finished, Available, Finished, False)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.
ds-copilot 0.1.25.2.28 requires pandas<3.0.0,>=1.5.0, but you have pandas 3.0.5 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed



In [2]:
# =============================================================================
# CELL 2 — Imports & Configuration
# =============================================================================
# Purpose : Load credentials and configure the 15-minute realtime poller.
#
# Schedule : Every 15 minutes via Fabric Pipeline
#
# Data fetched:
#   - Actual load (MW) per zone
#   - Generation mix by fuel type per zone
#   - Cross-border physical flows between zones
#
# Zones : Same 27 zones as daily poller
# =============================================================================

import time
import random
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
from entsoe import EntsoePandasClient
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType
)
from delta.tables import DeltaTable

# -----------------------------------------------------------------------------
# API Credentials — from Fabric Environment
# -----------------------------------------------------------------------------
ENTSOE_API_TOKEN = spark.conf.get("spark.pulsegrid.entsoe_token")
print(f"✅ ENTSO-E token loaded: {ENTSOE_API_TOKEN[:8]}...{ENTSOE_API_TOKEN[-4:]}")

# -----------------------------------------------------------------------------
# Rate Limiting
# -----------------------------------------------------------------------------
MAX_RETRIES    = 3
BACKOFF_FACTOR = 2
JITTER_MAX     = 1.5

# -----------------------------------------------------------------------------
# KQL Bronze Target
# -----------------------------------------------------------------------------
KUSTO_CLUSTER  = "https://trd-ratdj1p1b0yurnmn41.z4.kusto.fabric.microsoft.com"
KUSTO_DATABASE = "pulsegrid_bronze"

# -----------------------------------------------------------------------------
# ENTSO-E Zones — same as daily poller
# -----------------------------------------------------------------------------
ENTSOE_REGIONS = {
    "FR"   : "10YFR-RTE------C",
    "ES"   : "10YES-REE------0",
    "NL"   : "10YNL----------L",
    "BE"   : "10YBE----------2",
    "PL"   : "10YPL-AREA-----S",
    "AT"   : "10YAT-APG------L",
    "CH"   : "10YCH-SWISSGRIDZ",
    "PT"   : "10YPT-REN------W",
    "FI"   : "10YFI-1--------U",
    "CZ"   : "10YCZ-CEPS-----N",
    "SK"   : "10YSK-SEPS-----K",
    "HU"   : "10YHU-MAVIR----U",
    "RO"   : "10YRO-TEL------P",
    "BG"   : "10YCA-BULGARIA-R",
    "HR"   : "10YHR-HEP------M",
    "GR"   : "10YGR-HTSO-----Y",
    "SI"   : "10YSI-ELES-----O",
    "RS"   : "10YCS-SERBIATSOV",
    "LT"   : "10YLT-1001A0008Q",
    "LV"   : "10YLV-1001A00074",
    "DE-LU": "10Y1001A1001A82H",
    "IT-NO": "10Y1001A1001A73I",
    "DK-1" : "10YDK-1--------W",
    "DK-2" : "10YDK-2--------M",
    "SE-3" : "10Y1001A1001A46L",
    "NO-2" : "10YNO-2--------T",
    "EE"   : "10Y1001A1001A39I",
}

# -----------------------------------------------------------------------------
# Cross-border flow pairs to monitor
# -----------------------------------------------------------------------------
FLOW_PAIRS = [
    ("DE-LU", "FR",    "10Y1001A1001A82H", "10YFR-RTE------C"),
    ("FR",    "ES",    "10YFR-RTE------C",  "10YES-REE------0"),
    ("DE-LU", "NL",    "10Y1001A1001A82H", "10YNL----------L"),
    ("BE",    "FR",    "10YBE----------2",  "10YFR-RTE------C"),
    ("CH",    "DE-LU", "10YCH-SWISSGRIDZ", "10Y1001A1001A82H"),
    ("AT",    "DE-LU", "10YAT-APG------L",  "10Y1001A1001A82H"),
    ("CZ",    "DE-LU", "10YCZ-CEPS-----N",  "10Y1001A1001A82H"),
    ("PL",    "DE-LU", "10YPL-AREA-----S",  "10Y1001A1001A82H"),
    ("FR",    "BE",    "10YFR-RTE------C",  "10YBE----------2"),
    ("NL",    "BE",    "10YNL----------L",   "10YBE----------2"),
    ("NO-2",  "DK-1",  "10YNO-2--------T",  "10YDK-1--------W"),
    ("SE-3",  "DK-2",  "10Y1001A1001A46L",  "10YDK-2--------M"),
    ("FI",    "SE-3",  "10YFI-1--------U",   "10Y1001A1001A46L"),
    ("LT",    "LV",    "10YLT-1001A0008Q",  "10YLV-1001A00074"),
    ("LV",    "EE",    "10YLV-1001A00074",  "10Y1001A1001A39I"),
]

print(f"✅ Config loaded")
print(f"   Zones          : {len(ENTSOE_REGIONS)}")
print(f"   Flow pairs     : {len(FLOW_PAIRS)}")

StatementMeta(, c2d68a54-5456-404a-8387-b3e8c4e398ac, 10, Finished, Available, Finished, False)

✅ ENTSO-E token loaded: f3e904f3...6b27
✅ Config loaded
   Zones          : 27
   Flow pairs     : 15


In [3]:
# =============================================================================
# CELL 3 — Retry Helper + Generic KQL Writer (Fixed)
# =============================================================================
# Purpose : Retry only on transient errors — not on data availability errors.
#
# NoMatchingDataError — data doesn't exist for this time window
#   → No point retrying — skip immediately and try wide window
#
# Transient errors worth retrying:
#   → ConnectionError, TimeoutError, HTTPError (429/5xx)
#   → These may succeed on retry after a short wait
# =============================================================================

from entsoe.exceptions import NoMatchingDataError

# Errors that should NOT be retried — fail immediately
NO_RETRY_ERRORS = (NoMatchingDataError,)

def call_with_retry(fn, *args, **kwargs):
    """
    Execute fn with exponential backoff retry.
    Skips retries for NoMatchingDataError — data simply doesn't exist.
    Only retries transient network/server errors.
    """
    last_exception = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fn(*args, **kwargs)
        except NO_RETRY_ERRORS as e:
            # Data not available — no point retrying
            raise e
        except Exception as e:
            last_exception = e
            wait = (BACKOFF_FACTOR ** attempt) + random.uniform(0, JITTER_MAX)
            print(f"   ⚠️  Attempt {attempt}/{MAX_RETRIES} failed: {type(e).__name__} — retrying in {wait:.1f}s")
            time.sleep(wait)

    raise last_exception


def write_to_kql(df_spark, table_name):
    """Write Spark DataFrame to KQL Bronze table — append only."""
    df_spark.write \
        .format("com.microsoft.kusto.spark.datasource") \
        .option("kustoCluster",  KUSTO_CLUSTER) \
        .option("kustoDatabase", KUSTO_DATABASE) \
        .option("kustoTable",    table_name) \
        .option("accessToken",   mssparkutils.credentials.getToken("kusto")) \
        .mode("append") \
        .save()


def records_to_spark(records, schema):
    """Convert list of dicts to Spark DataFrame with Arrow fix."""
    pdf = pd.DataFrame(records)
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
    df  = spark.createDataFrame(pdf, schema=schema)
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    return df


print("✅ Retry helper defined — NoMatchingDataError skips retries immediately")

StatementMeta(, c2d68a54-5456-404a-8387-b3e8c4e398ac, 11, Finished, Available, Finished, False)

✅ Retry helper defined — NoMatchingDataError skips retries immediately


In [4]:
# =============================================================================
# CELL 4 — Actual Load Fetcher (Two-Stage Fetch)
# =============================================================================
# Purpose : Fetch actual electricity load (MW) for all zones.
#
# Two-stage fetch strategy:
#   Stage 1 — Narrow window (last 30 min):
#             Ideal for real-time — fetches only new data each cycle
#             Most zones publish within 15-30 minutes
#   Stage 2 — Wide window fallback (last 4 hours):
#             For slow TSOs with publication delays (CH, PT, RS, LV etc.)
#             Silver dedup ensures no duplicate storage
#
# Silver dedup safety net:
#   row_number() over (partition by region, event_time
#                      order by ingestion_time desc)
#   → Only latest ingestion of each record survives to Silver
# =============================================================================

LOAD_SCHEMA = StructType([
    StructField("ingestion_time", TimestampType(), False),
    StructField("event_time",     TimestampType(), False),
    StructField("region",         StringType(),    False),
    StructField("load_mw",        DoubleType(),    True),
    StructField("source",         StringType(),    True),
])

def fetch_load(region_code, zone_key, client, start, end):
    """Fetch actual load for one zone — handles both Series and DataFrame response."""
    load_data      = client.query_load(zone_key, start=start, end=end)
    records        = []
    ingestion_time = datetime.now(timezone.utc)

    # Handle both response formats
    if hasattr(load_data, 'columns'):
        col    = "Actual Load" if "Actual Load" in load_data.columns \
                 else load_data.columns[0]
        series = load_data[col]
    else:
        series = load_data

    for ts, load in series.items():
        if pd.notna(load):
            records.append({
                "ingestion_time": ingestion_time,
                "event_time"    : ts.to_pydatetime(),
                "region"        : region_code,
                "load_mw"       : float(load),
                "source"        : "ENTSO-E"
            })
    return records


def fetch_all_load(client, start_narrow, start_wide, end):
    """
    Fetch load for all zones using two-stage strategy.
    Stage 1: narrow 30-min window for fast TSOs
    Stage 2: wide 4-hour window fallback for slow TSOs
    """
    all_records = []
    failed      = []
    narrow_hits = 0
    wide_hits   = 0

    for region_code, zone_key in ENTSOE_REGIONS.items():
        try:
            # Stage 1 — narrow window
            try:
                records = call_with_retry(
                    fetch_load, region_code, zone_key,
                    client, start_narrow, end
                )
                if not records:
                    raise ValueError("Empty — trying wide window")
                narrow_hits += 1
            except:
                # Stage 2 — wide window fallback
                records = call_with_retry(
                    fetch_load, region_code, zone_key,
                    client, start_wide, end
                )
                wide_hits += 1

            all_records.extend(records)
            time.sleep(0.5 + random.uniform(0, 0.5))

        except Exception as e:
            failed.append(region_code)

    print(f"   ✅ Load fetched    : {len(all_records)} records")
    print(f"      Narrow window   : {narrow_hits} zones")
    print(f"      Wide window     : {wide_hits} zones")
    print(f"      Failed          : {failed if failed else 'None'}")
    return all_records

print("✅ Load fetcher defined — two-stage fetch strategy applied")

StatementMeta(, c2d68a54-5456-404a-8387-b3e8c4e398ac, 12, Finished, Available, Finished, False)

✅ Load fetcher defined — two-stage fetch strategy applied


In [5]:
# =============================================================================
# CELL 5 — Generation Mix Fetcher
# =============================================================================
# Purpose : Fetch actual generation by fuel type for all zones.
#           Returns one row per fuel type per zone per timestamp.
#           Solar + wind % are key ML features (suppress prices).
# =============================================================================

GENERATION_SCHEMA = StructType([
    StructField("ingestion_time", TimestampType(), False),
    StructField("event_time",     TimestampType(), False),
    StructField("region",         StringType(),    False),
    StructField("fuel_type",      StringType(),    True),
    StructField("generation_mw",  DoubleType(),    True),
    StructField("source",         StringType(),    True),
])

def fetch_generation(region_code, zone_key, client, start, end):
    """Fetch generation mix for one zone."""
    gen_data       = client.query_generation(zone_key, start=start, end=end)
    records        = []
    ingestion_time = datetime.now(timezone.utc)

    if hasattr(gen_data, 'columns'):
        for ts_idx, row in gen_data.iterrows():
            for fuel_type in gen_data.columns:
                val = row[fuel_type]
                if pd.notna(val):
                    records.append({
                        "ingestion_time": ingestion_time,
                        "event_time"    : ts_idx.to_pydatetime(),
                        "region"        : region_code,
                        "fuel_type"     : str(fuel_type),
                        "generation_mw" : float(val),
                        "source"        : "ENTSO-E"
                    })
    return records


def fetch_all_generation(client, start_narrow, start_wide, end):
    """
    Fetch generation mix for all zones using two-stage strategy.
    Stage 1: narrow 30-min window
    Stage 2: wide 4-hour window fallback
    """
    all_records = []
    failed      = []
    narrow_hits = 0
    wide_hits   = 0

    for region_code, zone_key in ENTSOE_REGIONS.items():
        try:
            # Stage 1 — narrow window
            try:
                records = call_with_retry(
                    fetch_generation, region_code, zone_key,
                    client, start_narrow, end
                )
                if not records:
                    raise ValueError("Empty — trying wide window")
                narrow_hits += 1
            except:
                # Stage 2 — wide window fallback
                records = call_with_retry(
                    fetch_generation, region_code, zone_key,
                    client, start_wide, end
                )
                wide_hits += 1

            all_records.extend(records)
            time.sleep(0.5 + random.uniform(0, 0.5))

        except Exception as e:
            failed.append(region_code)

    print(f"   ✅ Generation fetched : {len(all_records)} records")
    print(f"      Narrow window      : {narrow_hits} zones")
    print(f"      Wide window        : {wide_hits} zones")
    print(f"      Failed             : {failed if failed else 'None'}")
    return all_records

print("✅ Generation fetcher defined — two-stage fetch strategy applied")

StatementMeta(, c2d68a54-5456-404a-8387-b3e8c4e398ac, 13, Finished, Available, Finished, False)

✅ Generation fetcher defined — two-stage fetch strategy applied


In [6]:
# =============================================================================
# CELL 6 — Cross-Border Flow Fetcher
# =============================================================================
# Purpose : Fetch physical cross-border power flows between zone pairs.
#           Net import/export position is a strong price spike indicator.
#           Positive flow = power flowing from → to direction.
# =============================================================================

FLOW_SCHEMA = StructType([
    StructField("ingestion_time", TimestampType(), False),
    StructField("event_time",     TimestampType(), False),
    StructField("from_region",    StringType(),    False),
    StructField("to_region",      StringType(),    False),
    StructField("flow_mw",        DoubleType(),    True),
    StructField("source",         StringType(),    True),
])

def fetch_flows(from_code, to_code, from_key, to_key, client, start, end):
    """Fetch cross-border flows for one zone pair."""
    flow_data      = client.query_crossborder_flows(
        from_key, to_key, start=start, end=end
    )
    records        = []
    ingestion_time = datetime.now(timezone.utc)

    for ts, flow in flow_data.items():
        if pd.notna(flow):
            records.append({
                "ingestion_time": ingestion_time,
                "event_time"    : ts.to_pydatetime(),
                "from_region"   : from_code,
                "to_region"     : to_code,
                "flow_mw"       : float(flow),
                "source"        : "ENTSO-E"
            })
    return records


def fetch_all_flows(client, start_narrow, start_wide, end):
    """
    Fetch cross-border flows for all pairs using two-stage strategy.
    Stage 1: narrow 30-min window
    Stage 2: wide 4-hour window fallback
    """
    all_records = []
    failed      = []
    narrow_hits = 0
    wide_hits   = 0

    for from_code, to_code, from_key, to_key in FLOW_PAIRS:
        try:
            # Stage 1 — narrow window
            try:
                records = call_with_retry(
                    fetch_flows, from_code, to_code,
                    from_key, to_key, client, start_narrow, end
                )
                if not records:
                    raise ValueError("Empty — trying wide window")
                narrow_hits += 1
            except:
                # Stage 2 — wide window fallback
                records = call_with_retry(
                    fetch_flows, from_code, to_code,
                    from_key, to_key, client, start_wide, end
                )
                wide_hits += 1

            all_records.extend(records)
            time.sleep(0.5 + random.uniform(0, 0.5))

        except Exception as e:
            failed.append(f"{from_code}→{to_code}")

    print(f"   ✅ Flows fetched   : {len(all_records)} records")
    print(f"      Narrow window   : {narrow_hits} pairs")
    print(f"      Wide window     : {wide_hits} pairs")
    print(f"      Failed          : {failed if failed else 'None'}")
    return all_records

print("✅ Flow fetcher defined — two-stage fetch strategy applied")

StatementMeta(, c2d68a54-5456-404a-8387-b3e8c4e398ac, 14, Finished, Available, Finished, False)

✅ Flow fetcher defined — two-stage fetch strategy applied


In [7]:
# =============================================================================
# CELL 7 — Main Execution (Parallel Fetch)
# =============================================================================
# Purpose : Orchestrate the full 15-minute realtime poll cycle.
#           Fetches load, generation, and flows in PARALLEL using
#           ThreadPoolExecutor — all 3 run simultaneously.
#
# Parallelism Strategy:
#   - Each data type (load, generation, flows) runs in its own thread
#   - All 3 threads submit independent ENTSO-E API calls concurrently
#   - Total wall time ≈ slowest single fetch (not sum of all 3)
#   - Estimated time saving: ~60-70% vs sequential execution
#
# Thread Safety:
#   - Each thread uses its own EntsoePandasClient instance
#   - No shared mutable state between threads
#   - KQL writes are independent per table — no conflicts
#
# Schedule : Every 15 minutes via Fabric Pipeline
#
# Two-Stage Fetch Strategy:
#   Narrow (30 min) → fast TSOs
#   Wide (4 hours)  → slow TSOs fallback
#   Silver dedup    → handles overlapping records safely
#
# Daily API Call Budget:
#   Load       : 27 zones × 96 cycles = 2,592 calls/day
#   Generation : 27 zones × 96 cycles = 2,592 calls/day
#   Flows      : 15 pairs × 96 cycles = 1,440 calls/day
#   Total      : ~6,624 calls/day — 1.1% of ENTSO-E 400/min limit
# =============================================================================

from concurrent.futures import ThreadPoolExecutor, as_completed

cycle_start = datetime.now(timezone.utc)

print("=" * 55)
print("  PulseGrid — Realtime Poller (15-min, Parallel)")
print(f"  Run time: {cycle_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("=" * 55)

# -----------------------------------------------------------------------------
# Time Windows
# -----------------------------------------------------------------------------
now          = pd.Timestamp.now(tz="UTC")
start_narrow = now - pd.Timedelta(hours=2)    # narrow — most zones
start_wide   = now - pd.Timedelta(hours=6)    # wide — slow TSOs fallback
end          = now

print(f"\n  Time windows:")
print(f"   Narrow : {start_narrow.strftime('%H:%M')} → {end.strftime('%H:%M')} UTC (2 hours)")
print(f"   Wide   : {start_wide.strftime('%H:%M')} → {end.strftime('%H:%M')} UTC (6 hours)")

# -----------------------------------------------------------------------------
# Parallel Task Definitions
# Each task runs in its own thread with its own ENTSO-E client instance
# -----------------------------------------------------------------------------

def task_load():
    """Thread task — fetch and write actual load."""
    client  = EntsoePandasClient(api_key=ENTSOE_API_TOKEN)
    records = fetch_all_load(client, start_narrow, start_wide, end)

    if records:
        pdf = pd.DataFrame(records)
        pdf["ingestion_time"] = pd.to_datetime(pdf["ingestion_time"], utc=True)
        pdf["event_time"]     = pd.to_datetime(pdf["event_time"],     utc=True)
        pdf["region"]         = pdf["region"].astype(str)
        pdf["source"]         = pdf["source"].astype(str)
        pdf["load_mw"]        = pd.to_numeric(pdf["load_mw"], errors="coerce").astype("float64")

        spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
        df = spark.createDataFrame(pdf, schema=LOAD_SCHEMA)
        spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

        write_to_kql(df, "raw_electricity_load")

    return {"type": "load", "records": len(records)}


def task_generation():
    """Thread task — fetch and write generation mix."""
    client  = EntsoePandasClient(api_key=ENTSOE_API_TOKEN)
    records = fetch_all_generation(client, start_narrow, start_wide, end)

    if records:
        pdf = pd.DataFrame(records)
        pdf["ingestion_time"] = pd.to_datetime(pdf["ingestion_time"], utc=True)
        pdf["event_time"]     = pd.to_datetime(pdf["event_time"],     utc=True)
        pdf["region"]         = pdf["region"].astype(str)
        pdf["fuel_type"]      = pdf["fuel_type"].astype(str)
        pdf["source"]         = pdf["source"].astype(str)
        pdf["generation_mw"]  = pd.to_numeric(pdf["generation_mw"], errors="coerce").astype("float64")

        spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
        df = spark.createDataFrame(pdf, schema=GENERATION_SCHEMA)
        spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

        write_to_kql(df, "raw_generation_mix")

    return {"type": "generation", "records": len(records)}


def task_flows():
    """Thread task — fetch and write cross-border flows."""
    client  = EntsoePandasClient(api_key=ENTSOE_API_TOKEN)
    records = fetch_all_flows(client, start_narrow, start_wide, end)

    if records:
        pdf = pd.DataFrame(records)
        pdf["ingestion_time"] = pd.to_datetime(pdf["ingestion_time"], utc=True)
        pdf["event_time"]     = pd.to_datetime(pdf["event_time"],     utc=True)
        pdf["from_region"]    = pdf["from_region"].astype(str)
        pdf["to_region"]      = pdf["to_region"].astype(str)
        pdf["source"]         = pdf["source"].astype(str)
        pdf["flow_mw"]        = pd.to_numeric(pdf["flow_mw"], errors="coerce").astype("float64")

        spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
        df = spark.createDataFrame(pdf, schema=FLOW_SCHEMA)
        spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

        write_to_kql(df, "raw_cross_border_flows")

    return {"type": "flows", "records": len(records)}

# -----------------------------------------------------------------------------
# Execute all 3 tasks in parallel
# -----------------------------------------------------------------------------
print("\n🚀 Starting parallel fetch — Load + Generation + Flows simultaneously...")

tasks   = [task_load, task_generation, task_flows]
results = {}

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(task): task.__name__ for task in tasks}
    for future in as_completed(futures):
        try:
            result = future.result()
            results[result["type"]] = result["records"]
            print(f"   ✅ {result['type']:12} complete — {result['records']} records written")
        except Exception as e:
            task_name = futures[future]
            print(f"   ❌ {task_name} failed: {e}")
            results[task_name] = 0

# -----------------------------------------------------------------------------
# Cycle Summary
# -----------------------------------------------------------------------------
duration = (datetime.now(timezone.utc) - cycle_start).total_seconds()
total    = sum(results.values())

print(f"\n{'='*55}")
print(f"  Realtime Poller — Complete (Parallel)")
print(f"{'='*55}")
print(f"  Load records       : {results.get('load', 0)}")
print(f"  Generation records : {results.get('generation', 0)}")
print(f"  Flow records       : {results.get('flows', 0)}")
print(f"  Total written      : {total}")
print(f"  Duration           : {duration:.1f}s")
print(f"  Next run           : In 15 min (via Fabric Pipeline)")
print(f"{'='*55}")

StatementMeta(, c2d68a54-5456-404a-8387-b3e8c4e398ac, 15, Finished, Available, Finished, False)

  PulseGrid — Realtime Poller (15-min, Parallel)
  Run time: 2026-08-14 07:06:07 UTC

  Time windows:
   Narrow : 05:06 → 07:06 UTC (2 hours)
   Wide   : 01:06 → 07:06 UTC (6 hours)

🚀 Starting parallel fetch — Load + Generation + Flows simultaneously...
   ✅ Flows fetched   : 90 records
      Narrow window   : 14 pairs
      Wide window     : 0 pairs
      Failed          : ['SE-3→DK-2']
   ✅ Load fetched    : 120 records
      Narrow window   : 23 zones
      Wide window     : 4 zones
      Failed          : None
   ✅ Generation fetched : 1669 records
      Narrow window      : 22 zones
      Wide window        : 5 zones
      Failed             : None
   ✅ flows        complete — 90 records written
   ✅ load         complete — 120 records written
   ✅ generation   complete — 1669 records written

  Realtime Poller — Complete (Parallel)
  Load records       : 120
  Generation records : 1669
  Flow records       : 90
  Total written      : 1879
  Duration           : 207.3s
  Next run